In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]", "executable": "C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\tmp\\course-restructure-20261118\\scripts\\environment\\.venv-chem\\Scripts\\python.exe", "platform": "Windows-10-10.0.19045-SP0", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "colorama": "0.4.6", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0

# Разбор A. Область прогноза и идентифицируемость кинетической модели

Продолжаем анализ реальных оценок кофеина из [Table 4](https://doi.org/10.1186/s40064-015-0809-x). Перед расчётом сформулируйте критерий: прогноз проверяется на исключённой концентрации, а решение о переносе за 0.02–0.10 M требует нового эксперимента. Table 5 статьи содержит авторскую экстраполяцию на 30 °C; её нельзя добавлять как независимые измерения для обучения или проверки.

In [2]:
from pathlib import Path
import sys,json,math,csv
import numpy as np
from scipy import stats
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
OUT=workspace('M09-L02','case')
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=150,bbox_inches='tight')
    display(fig)
    plt.close(fig)


In [3]:
with (DATA/'caffeine_hydroxide_table4.csv').open(encoding='utf-8',newline='') as f:
    rows=list(csv.DictReader(f))
x=np.array([float(r['NaOH_mol_L']) for r in rows])
y=np.array([float(r['kexp_min_inverse']) for r in rows])
# Два столбца соответствуют свободному члену и зависимости от [OH-].
X=np.column_stack([np.ones(len(x)),x])
beta=np.linalg.lstsq(X,y,rcond=None)[0]
residual=y-X@beta
df=len(y)-X.shape[1]
inverse=np.linalg.inv(X.T@X)
s2=float(residual@residual/df)
covariance=s2*inverse
critical=stats.t.ppf(.975,df)
ci=np.column_stack([beta-critical*np.sqrt(np.diag(covariance)),beta+critical*np.sqrt(np.diag(covariance))])
print('intercept / min^-1; slope / L mol^-1 min^-1:',beta)
print('Conditional OLS intervals, NOT measured replicate uncertainty:',ci)


intercept / min^-1; slope / L mol^-1 min^-1: [0.0013 0.695 ]
Conditional OLS intervals, NOT measured replicate uncertainty: [[-0.02116501  0.02376501]
 [ 0.3563273   1.0336727 ]]


Диагональ матрицы H=X(XᵀX)⁻¹Xᵀ даёт leverage. Для линейного OLS остаток при исключении строки равен eᵢ/(1−hᵢ). Проверим эту формулу независимым переобучением пяти моделей, исключая целую концентрацию.

In [4]:
leverage=np.einsum('ij,jk,ik->i',X,inverse,X)
deleted_residual=residual/(1-leverage)
explicit=[]
for i in range(len(y)):
    keep=np.arange(len(y))!=i
    fitted=np.linalg.lstsq(X[keep],y[keep],rcond=None)[0]
    explicit.append(y[i]-X[i]@fitted)
assert np.allclose(deleted_residual,explicit,atol=1e-12)
rmse=float(np.sqrt(np.mean(deleted_residual**2)))
print('Leave-one-concentration-out RMSE / min^-1:',rmse)
query=np.array([.001,.03,.12])
Q=np.column_stack([np.ones(len(query)),query])
prediction=Q@beta
variance_mean=s2*np.einsum('ij,jk,ik->i',Q,inverse,Q)
half_width=critical*np.sqrt(s2+variance_mean)
display([{'NaOH_mol_L':float(c),'kexp':float(v),'conditional_prediction_interval':[float(v-w),float(v+w)],
          'domain':'interpolation' if min(x)<=c<=max(x) else 'extrapolation'}
         for c,v,w in zip(query,prediction,half_width)])
fig,ax=plt.subplots()
ax.errorbar(query,prediction,yerr=half_width,fmt='o',label='Conditional OLS prediction interval')
ax.scatter(x,y,label='Published estimates');ax.axhline(0,color='gray')
ax.set(xlabel='NaOH / mol L^-1',ylabel='k_exp / min^-1');ax.legend(fontsize=7)
publish_plot(fig,'prediction_domain.png')


Leave-one-concentration-out RMSE / min^-1: 0.008266125242857893


[{'NaOH_mol_L': 0.001,
  'kexp': 0.0019950000000000046,
  'conditional_prediction_interval': [-0.02882421571058901,
   0.03281421571058902],
  'domain': 'extrapolation'},
 {'NaOH_mol_L': 0.03,
  'kexp': 0.022150000000000006,
  'conditional_prediction_interval': [-0.0034192281362463925,
   0.047719228136246405],
  'domain': 'interpolation'},
 {'NaOH_mol_L': 0.12,
  'kexp': 0.08470000000000001,
  'conditional_prediction_interval': [0.05366013431803378,
   0.11573986568196624],
  'domain': 'extrapolation'}]

<Figure size 640x480 with 1 Axes>

Отрицательная нижняя граница показывает ограничение линейной модели и малой выборки; её нельзя молча обрезать и объявить физическим интервалом. Для позитивной модели нужен обоснованный закон ошибки и сравнение на новых данных. По этой таблице нельзя заново оценить Eₐ: она содержит одну температуру. Следующий опыт должен включать несколько температур, независимые приготовления, quench, проверку извлечения и отклика, а также случайный порядок анализа.

In [5]:
result={'kind':'calculated','analysis_scope':'published_data_reanalysis','source_doi':'10.1186/s40064-015-0809-x','slope_L_mol_inverse_min_inverse':float(beta[1]),'raw_time_series_refit':'NOT_RUN','scientific_review':'NOT_RUN'}
result['LOOCV_RMSE_min_inverse']=rmse


In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M09-L02','A',result,ROOT,inputs=['practicuum/shared/data/caffeine_hydroxide_table4.csv', 'practicuum/shared/data/caffeine_kinetics_provenance.json', 'practicuum/shared/data/source-responses/caffeine_kinetics.xml', 'practicuum/shared/data/source-responses/statin_stability_verified.xml'])
print(receipt)


C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\course-restructure-20261118\practicuum\evidence\20261119-professional-kinetics-windows-corrected\workspace\my_research\evidence\M09-L02\20261119T071830246005Z-A-7f5d71e02f9c4fc78bf136857660aeb6.json
